# Contextual Compression RAG — LangChain + Gemini + Sentence Transformers

**Goal:** Build a small RAG pipeline from an in-memory list of text documents. Retrieve with local `sentence-transformers` embeddings, compress retrieved documents *based on the question* using Google Gemini, and generate a grounded answer.

**Pipeline:** Documents → Embeddings → Vector Retrieval → `LLMChainExtractor` → Compressed Context → Answer.

> Requires a Google AI Studio Gemini API key and an internet connection for Gemini and the initial Sentence Transformers model download. Gemini calls may use paid API quota. Run cells in order.

## Install dependencies
Run the following once in Colab or Jupyter, then restart the kernel if requested.

In [ ]:
!pip install -q -U langchain-core langchain-classic langchain-google-genai langchain-huggingface langchain-text-splitters sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.0/268.0 kB 29.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.61.0 which is incompatible.


## Imports and Google API key
Get your key from [Google AI Studio](https://aistudio.google.com/apikey). The input hides the key; do **not** put it into code that you commit or share.

In [ ]:
import os

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [ ]:
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

## Tiny dataset in a Python list
Each entry intentionally mixes relevant and irrelevant details so we can see compression happen.

In [ ]:
data = [
    {"id": "doc1", "title": "RAG overview", "text": (
        "Retrieval-Augmented Generation (RAG) retrieves relevant information before generating an answer. "
        "A retriever may find several documents, but parts of those documents may be unrelated to the question. "
        "RAG systems may use citations to show where answers came from. "
        "The team holds a review meeting every Friday and uses blue slides.")},
    {"id": "doc2", "title": "Contextual compression", "text": (
        "Contextual compression filters or shortens retrieved documents using the user's question. "
        "An LLM-based extractor can preserve relevant sentences and remove irrelevant passages. "
        "This can reduce the amount of irrelevant context passed to the answer model. "
        "Our office has a coffee machine and a parking garage.")},
    {"id": "doc3", "title": "Embeddings", "text": (
        "Sentence Transformers converts text into dense vectors called embeddings. "
        "Similar meanings can have nearby vector representations. "
        "Cosine similarity is often used to compare vectors. "
        "The example project's logo is orange and its headquarters are in Cairo.")},
    {"id": "doc4", "title": "BM25 and hybrid search", "text": (
        "BM25 is a lexical ranking algorithm using term frequency, inverse document frequency, and document length normalization. "
        "Hybrid retrieval combines lexical and semantic retrieval. "
        "The annual picnic is planned for September and the team prefers tea.")},
    {"id": "doc5", "title": "Reranking", "text": (
        "A cross-encoder evaluates a query and a candidate document jointly and can rerank retrieved candidates. "
        "Reranking improves candidate ordering but is not the same as extracting relevant passages. "
        "The company's cafeteria serves pizza on Tuesdays.")},
]

documents = [Document(page_content=x["text"], metadata={"id": x["id"], "title": x["title"]}) for x in data]
print(f"Loaded {len(documents)} documents")

Loaded 5 documents


## Sentence Transformers embeddings + vector retriever
The local model downloads once and then runs on your machine. `InMemoryVectorStore` is enough for this small demonstration.

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = InMemoryVectorStore.from_documents(documents, embedding=embeddings)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
query = "How does contextual compression improve RAG retrieval results?"
# query = "Who is Mohamed Salah"

retrieved_docs = base_retriever.invoke(query)

print("BEFORE COMPRESSION:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n[{i}] {doc.metadata['title']} ({len(doc.page_content)} chars)\n{doc.page_content}")

BEFORE COMPRESSION:

[1] RAG overview (329 chars)
Retrieval-Augmented Generation (RAG) retrieves relevant information before generating an answer. A retriever may find several documents, but parts of those documents may be unrelated to the question. RAG systems may use citations to show where answers came from. The team holds a review meeting every Friday and uses blue slides.

[2] Contextual compression (307 chars)
Contextual compression filters or shortens retrieved documents using the user's question. An LLM-based extractor can preserve relevant sentences and remove irrelevant passages. This can reduce the amount of irrelevant context passed to the answer model. Our office has a coffee machine and a parking garage.

[3] Reranking (244 chars)
A cross-encoder evaluates a query and a candidate document jointly and can rerank retrieved candidates. Reranking improves candidate ordering but is not the same as extracting relevant passages. The company's cafeteria serves pizza on Tuesdays.


## Gemini as an LLM-based document compressor
`LLMChainExtractor` extracts question-relevant spans; it is different from simple token truncation or reranking. Compression may also remove entire documents that lack useful spans.

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash", temperature=0)

compressor = LLMChainExtractor.from_llm(llm)
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor,
)

compressed_docs = compression_retriever.invoke(query)
print("AFTER COMPRESSION:")
for i, doc in enumerate(compressed_docs, 1):
    print(f"\n[{i}] {doc.metadata.get('title', 'Untitled')} ({len(doc.page_content)} chars)\n{doc.page_content}")

AFTER COMPRESSION:

[1] Contextual compression (253 chars)
Contextual compression filters or shortens retrieved documents using the user's question. An LLM-based extractor can preserve relevant sentences and remove irrelevant passages. This can reduce the amount of irrelevant context passed to the answer model.


## Compare before vs after


In [ ]:
before_chars = sum(len(doc.page_content) for doc in retrieved_docs)
after_chars = sum(len(doc.page_content) for doc in compressed_docs)

print(f"Retrieved documents: {len(retrieved_docs)} | Compressed documents: {len(compressed_docs)}")
print(f"Total characters: {before_chars} → {after_chars}")
print(f"Change: {(1 - after_chars / before_chars) * 100:.1f}%" if before_chars else "No documents retrieved")

Retrieved documents: 3 | Compressed documents: 1
Total characters: 880 → 253
Change: 71.2%


## Generate a grounded answer from compressed context

In [ ]:
context = "\n\n".join(
    f"Source: {doc.metadata.get('title', 'Unknown')}\n{doc.page_content}"
    for doc in compressed_docs
)
prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY the provided context. If the information is missing, say you don't know. Keep it concise and cite source titles in parentheses."),
    ("human", "Question: {question}\n\nContext:\n{context}"),
])

answer_chain = prompt | llm | StrOutputParser()
answer = answer_chain.invoke({"question": query, "context": context}) if context else "No relevant context returned."
print(answer)

Contextual compression improves RAG retrieval results by filtering or shortening retrieved documents based on the user's question, preserving relevant sentences and removing irrelevant passages to reduce the amount of irrelevant context passed to the answer model (Contextual compression).
